In [1]:
import pandas as pd


ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')


print("Ledger shape:", ledger.shape)
print("Gateway shape:", gateway.shape)

ledger.head()

Ledger shape: (10, 6)
Gateway shape: (9, 6)


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card


In [2]:
print(ledger.columns)
print(gateway.columns)


Index(['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd',
       'status', 'payment_method'],
      dtype='object')
Index(['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd',
       'status', 'payment_method'],
      dtype='object')


In [3]:

print("Ledger duplicate rows:", ledger.duplicated().sum())
print("Gateway duplicate rows:", gateway.duplicated().sum())


print("\nLedger nulls:\n", ledger.isnull().sum())
print("\nGateway nulls:\n", gateway.isnull().sum())

Ledger duplicate rows: 0
Gateway duplicate rows: 0

Ledger nulls:
 transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

Gateway nulls:
 transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


In [4]:

missing_in_gateway = ledger[~ledger['transaction_id'].isin(gateway['transaction_id'])]


missing_in_ledger = gateway[~gateway['transaction_id'].isin(ledger['transaction_id'])]

print("Missing in Gateway:", len(missing_in_gateway))
print("Missing in Ledger:", len(missing_in_ledger))

Missing in Gateway: 2
Missing in Ledger: 1


In [5]:
missing_in_gateway.to_csv('missing_in_gateway.csv', index=False)
missing_in_ledger.to_csv('.missing_in_ledger.csv', index=False)

In [6]:

merged = pd.merge(
    ledger,
    gateway,
    on='transaction_id',
    suffixes=('_ledger', '_gateway')
)

print("Total matched records:", len(merged))

Total matched records: 8


In [7]:
amount_mismatches = merged[
    merged['amount_usd_ledger'] != merged['amount_usd_gateway']
]

print("Amount mismatches:", len(amount_mismatches))

Amount mismatches: 2


In [8]:
status_mismatches = merged[
    merged['status_ledger'] != merged['status_gateway']
]

print("Status mismatches:", len(status_mismatches))

Status mismatches: 1


8 matched
2 missing in gateway
1 missing in ledger
Everything adds up correctly:

In [9]:
amount_mismatches.to_csv('amount_mismatches.csv', index=False)
status_mismatches.to_csv('status_mismatches.csv', index=False)

In [10]:

missing_in_gateway['issue'] = 'missing_in_gateway'
missing_in_ledger['issue'] = 'missing_in_ledger'
amount_mismatches['issue'] = 'amount_mismatch'
status_mismatches['issue'] = 'status_mismatch'


reconciliation_report = pd.concat([
    missing_in_gateway,
    missing_in_ledger,
    amount_mismatches,
    status_mismatches
], ignore_index=True)

print("Total reconciliation issues:", len(reconciliation_report))

Total reconciliation issues: 6


/tmp/ipykernel_17615/2092659288.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_in_gateway['issue'] = 'missing_in_gateway'
/tmp/ipykernel_17615/2092659288.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_in_ledger['issue'] = 'missing_in_ledger'
/tmp/ipykernel_17615/2092659288.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pyda

In [11]:
reconciliation_report.to_csv('reconciliation_report.csv', index=False)

In [12]:
summary_metrics = {
    "total_ledger_rows": len(ledger),
    "total_gateway_rows": len(gateway),
    "missing_in_gateway_count": len(missing_in_gateway),
    "missing_in_ledger_count": len(missing_in_ledger),
    "amount_mismatch_count": len(amount_mismatches),
    "status_mismatch_count": len(status_mismatches),
    "reconciliation_issue_count": len(reconciliation_report),
    "ledger_total_amount": ledger['amount_usd'].sum(),
    "gateway_total_amount": gateway['amount_usd'].sum(),
    "amount_at_risk": amount_mismatches['amount_usd_ledger'].sum()
}

import json

with open('summary_metrics.json', 'w') as f:
    json.dump(summary_metrics, f, indent=4)

summary_metrics

{'total_ledger_rows': 10,
 'total_gateway_rows': 9,
 'missing_in_gateway_count': 2,
 'missing_in_ledger_count': 1,
 'amount_mismatch_count': 2,
 'status_mismatch_count': 1,
 'reconciliation_issue_count': 6,
 'ledger_total_amount': np.float64(23340.0),
 'gateway_total_amount': np.float64(20550.0),
 'amount_at_risk': np.float64(1490.0)}

JSON Normalization

In [23]:
import json
import pandas as pd


with open('api_response_sample.json') as f:
    data = json.load(f)

records = []

for batch in data['batches']:
    merchant = batch['merchant']

    for s in batch['settlements']:
        records.append({
            'batch_id': batch['batch_id'],
            'merchant_id': merchant['merchant_id'],
            'merchant_name': merchant['merchant_name'],
            'region': merchant['region'],
            'settlement_id': s['settlement_id'],
            'amount_usd': s['amount_usd'],
            'status': s['status'],
            'processed_at': s['processed_at'],
            'bank_name': s['bank']['name'],
            'bank_country': s['bank']['country']
        })


df_api = pd.DataFrame(records)
df_api['processed_at'] = pd.to_datetime(df_api['processed_at']).dt.date
#df_api.head(10)
df_api.to_csv('api_normalized.csv', index=False)

for dashboard


In [18]:
df_api['processed_at'] = pd.to_datetime(df_api['processed_at']).dt.date

daily_summary = df_api.groupby('processed_at').agg(
    total_amount=('amount_usd', 'sum'),
    total_transactions=('settlement_id', 'count')
).reset_index()

daily_summary.to_csv('daily_summary.csv', index=False)

,processed_at,total_amount,total_transactions
0,2026-03-07,12940.5,6


In [20]:
region_breakdown = df_api.groupby('region').agg(
    total_amount=('amount_usd', 'sum'),
    total_transactions=('settlement_id', 'count')
).reset_index()

region_breakdown.to_csv('region_breakdown.csv', index=False)

In [21]:
merchant_performance = df_api.groupby('merchant_name').agg(
    total_amount=('amount_usd', 'sum'),
    total_transactions=('settlement_id', 'count')
).reset_index()

merchant_performance.to_csv('merchant_performance_summary.csv', index=False)

In [25]:

df_txn = pd.read_csv('cleaned_transactions.csv')


payment_method_breakdown = df_txn.groupby('payment_method').agg(
    total_amount=('amount_usd', 'sum'),
    total_transactions=('transaction_id', 'count')
).reset_index()


payment_method_breakdown.to_csv('payment_method_breakdown.csv', index=False)

payment_method_breakdown

,payment_method,total_amount,total_transactions
0,Card,52972.0,13
1,NetBanking,15306.0,3
2,UPI,34397.5,9
3,Wallet,13404.5,5
